In [3]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


In [ ]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag: Literal['positive', 'negative']

llm_structured_output = llm.with_structured_output(llm_schema)


llm_schema(movie_summary_flag='negative')

In [4]:
# Task 1 - Prompt
prompt_template = ChatPromptTemplate.from_messages([
    ('system', "You are a movie review evaluator."),
    ('human', "Please categorise the movie review as positive or negative : {input}")
])

In [8]:
#Task 2 - Custom Runnable
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableBranch

def pydantic_json(input: llm_schema) -> str:
    return input.model_dump()['movie_summary_flag']

pydantic_json_runnable = RunnableLambda(pydantic_json)

Conditional chain 1

In [6]:
linkedin_prompt = ChatPromptTemplate.from_messages([
   ("system", "You are a linkedin post generator."),
   ("human", "Generate a linkedin post for the following content : {text}")
])

chain_linkedin = linkedin_prompt | llm 

Conditional Parallel Chain 2

In [7]:
def insta_chain(text: dict):
    insta_prompt = ChatPromptTemplate.from_messages([
      ("system", "You are a Instagram post generator."),
      ("human", "Generate a Instagram post for the following content : {text}")
    ])

    chain_insta = insta_prompt | llm 

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

Final

In [12]:
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),
    insta_chain_runnable
)

final_chain = prompt_template | llm_structured_output | pydantic_json_runnable | conditional_chain

In [13]:
final_chain.invoke("I loved the hail mary movie")

AIMessage(content='Okay, "positive" is a fantastic starting point! To make it a great LinkedIn post, we need to give it some context. Here are a few options, ranging from general inspiration to more specific professional scenarios. Choose the one that best fits your current situation, or mix and match!\n\n---\n\n**Option 1: General Inspirational Positivity**\n\n**Post:**\n"There\'s immense power in cultivating a positive mindset, especially in our professional lives. It\'s not about ignoring challenges, but approaching them with optimism and a belief in solutions. ✨\n\nI truly believe that a positive outlook can transform obstacles into opportunities, foster better collaboration, and ultimately drive greater success. Let\'s uplift each other and spread that positive energy!\n\nWhat\'s one small thing you do to maintain a positive perspective during your workday? Share below! 👇\n\n#PositiveVibes #MindsetMatters #ProfessionalGrowth #Inspiration #WorkLife"\n\n---\n\n**Option 2: Celebratin

In [12]:
def beautify(final_response: dict) -> dict:
    linkedin_response = final_response['branches']['linkedin']
    insta_response = final_response['branches']['insta']
    return {"Linkedin": linkedin_response, "Instagram": insta_response}
beautify_runnable = RunnableLambda(beautify)

beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("Oppenheimer")
    

{'Linkedin': 'Here are a few options for a LinkedIn post about "Oppenheimer," playing on different angles:\n\n---\n\n**Option 1 (Focus on Leadership & Ethics):**\n\nBeyond the cinematic spectacle, "Oppenheimer" offers a profound case study in leadership, innovation, and ethical responsibility.\n\nThe film meticulously chronicles J. Robert Oppenheimer\'s journey leading the top-secret Manhattan Project, from the immense scientific ambition to the devastating reality of the Trinity test and its aftermath. It\'s a powerful exploration of the moral reckoning faced by the "father of the atomic bomb" and his subsequent fall from grace due to political machinations.\n\nThis story isn\'t just history; it\'s a stark reminder of the complex interplay between scientific advancement, ethical dilemmas, and political power.\n\nWhat leadership lessons or ethical considerations did you take away from Oppenheimer\'s story? Share your thoughts below. 👇\n\n#Oppenheimer #Leadership #Ethics #Innovation #Hi